# MKWii RL Training Monitor
Run the cell below to start training and monitor episode rewards in real time.

In [ ]:
import subprocess, sys, os

RUNS_DIR = os.path.join(os.path.dirname(os.getcwd()), "runs")
tb_proc = subprocess.Popen(
    [sys.executable, "-m", "tensorboard.main", "--logdir", RUNS_DIR, "--port", "6006"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"TensorBoard running at http://localhost:6006  (logdir: {RUNS_DIR})")
print("Run the cell below to start training. Stop this cell to shut down TensorBoard.")

In [ ]:
import subprocess, sys, re, os
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display
import ipywidgets as widgets
import json

import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(f'You are training on your {torch.cuda.get_device_name(0)}.')
else:
    print('No GPU detected. You are training on a CPU. Training will be very slow.')

matplotlib.rcParams['figure.figsize'] = (12, 5)

PROJECT_ROOT = os.path.dirname(os.getcwd())
START_SCRIPT = os.path.join(PROJECT_ROOT, "scripts", "train", "start_training.py")
STATE_FILE = os.path.join(PROJECT_ROOT, "scripts", "training_state.json")

try:
    with open(STATE_FILE) as f:
        episode_offset = json.load(f).get("episode_count", 0)
except Exception:
    episode_offset = 0

# Storage for episode data
p1_rewards = []
p2_rewards = []
p1_episodes = []
p2_episodes = []
episode_count = 0

PATTERN = re.compile(r'\[TrainingProcess\] P(\d) episode \d+ end\. stuck=(\w+) total_reward=([\-\d\.]+)')

plot_output = widgets.Output()
display(plot_output)

def update_plot():
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax1 = plt.subplots(1, 1)

        if p1_rewards:
            ax1.plot(p1_episodes, p1_rewards, 'o-', color='#00E5FF', label='P1', linewidth=1.5, markersize=4)
        if p2_rewards:
            ax1.plot(p2_episodes, p2_rewards, 'o-', color='#FF6B6B', label='P2', linewidth=1.5, markersize=4)
        ax1.axhline(y=0, color='white', linestyle='--', alpha=0.3)
        ax1.set_title('Episode Total Reward', color='white')
        ax1.set_xlabel('Episode', color='white')
        ax1.set_ylabel('Total Reward', color='white')
        ax1.legend()
        ax1.set_facecolor('#1a1a2e')
        fig.patch.set_facecolor('#0f0f23')
        ax1.tick_params(colors='white')
        ax1.spines['bottom'].set_color('white')
        ax1.spines['left'].set_color('white')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.show()

def parse_line(line):
    global episode_count
    line = line.strip()
    if not line:
        return
    print(line, flush=True)
    matches = PATTERN.findall(line)
    for m in matches:
        player = int(m[0])
        reward = float(m[2])
        if player == 1:
            p1_rewards.append(reward)
        else:
            p2_rewards.append(reward)

    new_count = min(len(p1_rewards), len(p2_rewards))
    if new_count > episode_count:
        for i in range(episode_count + 1, new_count + 1):
            p1_episodes.append(episode_offset + i)
            p2_episodes.append(episode_offset + i)
        episode_count = new_count
        update_plot()

print(f"Starting training from: {START_SCRIPT}")
proc = subprocess.Popen(
    [sys.executable, "-u", START_SCRIPT],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        parse_line(line)
except KeyboardInterrupt:
    proc.terminate()
    print("Training stopped.")

proc.wait()
print("Training process exited.")

## Manual Model Save
Run the cell below to snapshot the current model, buffer, TensorBoard logs, and training state into `runs_backup/run{N}/`.

In [ ]:
import os, shutil

PROJECT_ROOT = os.path.normpath(os.path.join(os.getcwd(), ".."))
BACKUP_DIR   = os.path.join(PROJECT_ROOT, "runs_backup")

# Determine next run number
existing = [int(d[3:]) for d in os.listdir(BACKUP_DIR) if d.startswith("run") and d[3:].isdigit()]
run_num  = max(existing, default=0) + 1
dst      = os.path.join(BACKUP_DIR, f"run{run_num}")
os.makedirs(dst, exist_ok=True)

# Copy model + buffer
for fname in ("agent_model.pth", "agent_model_buffer.pkl"):
    src = os.path.join(PROJECT_ROOT, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(dst, fname))
        print(f"Copied {fname}")
    else:
        print(f"MISSING: {fname}")

# Copy TensorBoard runs/
runs_src = os.path.join(PROJECT_ROOT, "runs")
if os.path.exists(runs_src):
    shutil.copytree(runs_src, os.path.join(dst, "runs"), dirs_exist_ok=True)
    print("Copied runs/")
else:
    print("MISSING: runs/")

# Copy training_state.json
state_src = os.path.join(PROJECT_ROOT, "scripts", "training_state.json")
if os.path.exists(state_src):
    scripts_dst = os.path.join(dst, "scripts")
    os.makedirs(scripts_dst, exist_ok=True)
    shutil.copy2(state_src, os.path.join(scripts_dst, "training_state.json"))
    print("Copied scripts/training_state.json")
else:
    print("MISSING: scripts/training_state.json")

# Copy crash log
crash_src = os.path.join(PROJECT_ROOT, "crash_log.txt")
if os.path.exists(crash_src):
    shutil.copy2(crash_src, os.path.join(dst, "crash_log.txt"))
    print("Copied crash_log.txt")
else:
    print("MISSING: crash_log.txt (ok if no crashes)")

print(f"Snapshot saved to runs_backup/run{run_num}/")
